# Multi-Agent Research Report Generator (CrewAI + Gemini)

This notebook builds an automated "newsroom" of AI agents that work together to research, write, fact-check, humanize, combine, and publish a long-form report on a topic of your choice.

We use the **CrewAI** framework, which lets you define a team of `Agent`s (each with a role, a goal, and a backstory that shapes how it "thinks"), give them `Task`s to complete, and wire those tasks together into a `Crew` that runs them in order.

The pipeline we build has **nine agents**:

1. A **Research & Evidence Analyst** who gathers facts, statistics, recent examples, and case studies.
2. An **Introduction Writer**.
3. A **Core Argument Analyst** who states the report's central thesis.
4. A **Multi-Dimensional Analyst** who explores the issue from several angles (environmental, economic, social, political, etc.).
5. A **Counter-Argument & Critique Specialist** who raises and answers objections.
6. A **Synthesis & Conclusion Writer** who ties everything together and makes recommendations.
7. A **Fact-Checker & Humanizing Editor** who independently reviews *each* of the sections above against the evidence and rewrites them using the style rules in `skills/humanizer/SKILL.md`.
8. A **Managing Editor** who combines all the verified sections into one cohesive report.
9. A **Publishing Specialist** who saves the final report as a Word (`.docx`) document using a custom tool.

As a worked example, we'll ask the crew to write a report on **environmental security, climate-change-driven disasters, and their consequences, with a particular focus on India** - but you can change the topic in the "Report inputs" section below and rerun everything.

By default this notebook uses **Google Gemini** as the underlying language model (via CrewAI's native Gemini provider), because it's fast and inexpensive. Later in the notebook we show, in a single clearly marked cell, how to swap in OpenAI or Groq instead - the rest of the notebook does not need to change.

## 1. Prerequisites

This notebook depends on a few Python packages on top of the ones already used in this project:

- `crewai` - the multi-agent orchestration framework (already a project dependency).
- `google-genai` - Google's Gen AI SDK, which CrewAI uses under the hood to talk to Gemini models.
- `python-docx` - used by our custom tool to write the final report out as a `.docx` Word document.
- `python-dotenv` - used to load API keys from the `.env` file (already a project dependency).

You also need a `GOOGLE_API_KEY` in your `.env` file (this project's `.env` already has one). If you'd rather use OpenAI or Groq, the same `.env` file should contain `OPENAI_API_KEY` or `GROQ_API_KEY`.

Run the cell below once to make sure `google-genai` and `python-docx` are installed in this kernel's environment. If they are already installed, `pip` will simply confirm that the requirement is already satisfied. If this cell fails because files are locked, close any other running notebook kernels in this project, restart this kernel, and try again - or install the packages from a terminal with `uv add google-genai python-docx`.

In [ ]:
%pip install -q google-genai python-docx

## 2. Imports

Here is what each import is for:

- `os` and `pathlib.Path` - reading environment variables and working with file paths (for the `.env` file, the skill file, and the output `.docx` file).
- `dotenv.load_dotenv` - loads the key/value pairs from your `.env` file into the process's environment variables, so `os.getenv("GOOGLE_API_KEY")` can find them.
- `crewai.Agent`, `Task`, `Crew`, `Process`, `LLM` - the core building blocks: `Agent` defines a "persona", `Task` defines a unit of work for an agent, `Crew` groups agents and tasks together and runs them, `Process` controls the running order, and `LLM` wraps whichever language model we choose to use.
- `crewai.tools.tool` - a decorator that turns an ordinary Python function into a tool an agent can call.
- `docx.Document` - the class from `python-docx` that represents a Word document we can build up and save.
- `IPython.display.Markdown` and `display` - used at the end of the notebook to render the final report nicely instead of as raw text.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from docx import Document
from IPython.display import Markdown, display

## 3. Load environment variables and API keys

`load_dotenv(override=True)` reads the `.env` file in this project's root folder and copies its values into the environment variables that this Python process can see. Passing `override=True` means that if a variable is already set elsewhere (for example in your shell), the value from `.env` will take priority - this keeps the notebook's behaviour predictable.

We then read the three API keys this project's `.env` file provides. We only *require* `GOOGLE_API_KEY` for this notebook's default setup, but we read the other two as well so the "switch provider" cell later on can use them without any extra setup.

In [2]:
load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if not google_api_key:
    raise ValueError(
        "GOOGLE_API_KEY is missing. Add it to your .env file before running this notebook."
    )

print(f"Google API key loaded (starts with {google_api_key[:8]}...)")

Google API key loaded (starts with AQ.Ab8RN...)


## 4. Choose the language model

CrewAI's `LLM` class is a thin wrapper that can talk to many different providers. You tell it which model to use with a string of the form `"<provider>/<model-name>"` (for some providers, like OpenAI, the provider prefix can be left off because it's the default).

For this notebook we use `"gemini/gemini-2.0-flash"`, a fast and low-cost Gemini model. We also set `temperature=0.4`, which is a number between 0 and 1 (sometimes higher) that controls how "creative" versus "predictable" the model's text is - lower values make the model stick closer to the most likely wording, which is a reasonable default for a research report.

Every `Agent` we create later will be given this same `llm` object, so if you want to change the model for the *entire* crew, you only need to edit this one cell. The commented-out lines show how you would point the exact same notebook at OpenAI or Groq instead - CrewAI handles the differences between providers internally, so no other code needs to change.

In [3]:
GEMINI_MODEL = "gemini/gemini-3.5-flash"

llm = LLM(
    model=GEMINI_MODEL,
    api_key=google_api_key,
    temperature=0.2,
)

print(f"This crew will use: {GEMINI_MODEL}")

# --- To use a different provider instead, comment out the block above and ---
# --- uncomment one of the blocks below. Everything else in the notebook   ---
# --- stays exactly the same.                                              ---

# OpenAI example:
# llm = LLM(model="gpt-4o-mini", api_key=openai_api_key, temperature=0.4)

# Groq example:
# llm = LLM(model="groq/llama-3.3-70b-versatile", api_key=groq_api_key, temperature=0.4)

This crew will use: gemini/gemini-3.5-flash


## 5. Define the report's topic, audience, and tone

CrewAI lets you write `Task` descriptions as templates containing placeholders like `{topic}`, and then fill them in later by passing an `inputs` dictionary to `crew.kickoff(...)`. This means we can write all of our task descriptions once, using `{topic}`, `{audience}`, and `{tone}`, and then generate a report on a *different* subject just by changing the dictionary below and re-running the crew - no need to edit every task.

For our worked example we'll focus on **environmental security and climate-change-driven disasters in India**, but feel free to change `topic` to anything you like.

In [4]:
report_inputs = {
    "topic": "Environmental Security in India",
    "audience": "policy analysts, journalists, and informed general readers",
    "tone": "clear, engaging, and natural, never robotic or repetitive",
    "context": "AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security"
}

report_inputs

{'topic': 'Environmental Security in India',
 'audience': 'policy analysts, journalists, and informed general readers',
 'tone': 'clear, engaging, and natural, never robotic or repetitive',
 'context': 'AI infra requirements, climate change, rainfed agri, groundwater extraction, and food security'}

## 6. Load the humanizer skill instructions

The "Fact-Checker & Humanizing Editor" agent we'll create later needs to know *how* to humanize text. Rather than hard-coding those rules into this notebook, we keep them in a separate Markdown file, `skills/humanizer/SKILL.md`, which describes both a fact-checking checklist and a list of writing-style rules for making AI-drafted text sound more natural.

Keeping these instructions in their own file means you can tune the editor's behaviour - for example, make it stricter about fact-checking, or change the target writing style - without touching this notebook at all. Here we simply read the file's contents into a string so we can embed it in that agent's `backstory` further down.

In [5]:
humanizer_skill_path = Path("skills/SKILL.md")
humanizer_instructions = humanizer_skill_path.read_text(encoding="utf-8")

print(f"Loaded {len(humanizer_instructions)} characters of humanizer instructions from {humanizer_skill_path}")
print("---")
print(humanizer_instructions[:400] + "...")

Loaded 27675 characters of humanizer instructions from skills\SKILL.md
---
---
name: humanizer
version: 2.5.1
description: |
  Remove signs of AI-generated writing from text. Use when editing or reviewing
  text to make it sound more natural and human-written. Based on Wikipedia's
  comprehensive "Signs of AI writing" guide. Detects and fixes patterns including:
  inflated symbolism, promotional language, superficial -ing analyses, vague
  attributions, em dash overuse, ...


## 7. A custom tool: saving the report as a Word document

Agents in CrewAI can only directly produce *text*. To get a `.docx` file out of the pipeline, we give one agent access to a **tool**: an ordinary Python function that the agent can choose to call, passing it arguments.

The `@tool` decorator from `crewai.tools` turns a Python function into a tool. Two things about the function matter for how the agent uses it:

- Its **docstring** becomes the description the agent reads to decide when and how to use the tool, so it needs to clearly explain what the tool does and what each argument means.
- Its **type hints** (`report_markdown: str`, `file_name: str`, return type `str`) tell CrewAI what arguments the tool expects and what kind of value it returns.

The tool itself walks through the report text line by line. Because we've been asking every writing agent to produce Markdown, we can use simple rules to convert that Markdown into Word formatting with `python-docx`:

- A line starting with `# `, `## `, or `### ` becomes a Word heading of level 1, 2, or 3.
- A line starting with `- ` or `* ` becomes a bulleted list item.
- A line starting with `>` becomes a quote-styled paragraph (used for the fact-checker's optional `> Note:` lines).
- Any other non-empty line becomes a normal paragraph.
- Within any line, text wrapped in `**double asterisks**` is rendered in bold, by splitting the line on `**` and alternating between normal and bold "runs" of text.

In [6]:
@tool("save_report_as_word_document")
def save_report_as_word_document(report_markdown: str, file_name: str) -> str:
    """Save a research report written in simple Markdown as a formatted Microsoft Word (.docx) file.

    The report_markdown text may use '#', '##' and '###' at the start of a line for
    headings, '-' or '*' for bullet points, '>' for a short quoted note, and
    '**bold text**' for bold emphasis. Every other non-empty line is treated as a
    normal paragraph.

    Args:
        report_markdown: The full text of the report, in Markdown format.
        file_name: The file name to save the document as, e.g. 'report.docx'.

    Returns:
        A short confirmation message containing the absolute path of the saved file.
    """
    document = Document()

    def add_text_with_bold(paragraph, text):
        # Splitting on "**" means every second chunk (index 1, 3, 5, ...) was
        # originally wrapped in "**bold**" markers, so we render it as bold.
        for index, chunk in enumerate(text.split("**")):
            if chunk == "":
                continue
            run = paragraph.add_run(chunk)
            run.bold = index % 2 == 1

    for raw_line in report_markdown.splitlines():
        line = raw_line.strip()

        if not line:
            continue
        elif line.startswith("### "):
            document.add_heading(line[4:].strip(), level=3)
        elif line.startswith("## "):
            document.add_heading(line[3:].strip(), level=2)
        elif line.startswith("# "):
            document.add_heading(line[2:].strip(), level=1)
        elif line.startswith("- ") or line.startswith("* "):
            paragraph = document.add_paragraph(style="List Bullet")
            add_text_with_bold(paragraph, line[2:].strip())
        elif line.startswith(">"):
            paragraph = document.add_paragraph(style="Intense Quote")
            add_text_with_bold(paragraph, line.lstrip("> ").strip())
        else:
            paragraph = document.add_paragraph()
            add_text_with_bold(paragraph, line)

    output_path = Path(file_name).resolve()
    document.save(output_path)

    return f"Report saved to {output_path}"

## 8. Defining the agents

Every `Agent` in CrewAI is configured mainly through three fields, which are combined into the prompt the language model sees whenever that agent is asked to do something:

- **`role`** - a short job title, e.g. "Introduction Writer".
- **`goal`** - what the agent is trying to achieve, written as an instruction.
- **`backstory`** - a paragraph of context that shapes the agent's "voice" and priorities. This is also where we'll embed the humanizer skill instructions for the fact-checking agent.

We pass `llm=llm` to every agent so they all use the model we configured in Step 4, and `verbose=True` so that, while the crew is running, you can see what each agent is doing in the notebook's output.

Notice that none of the text below mentions the specific report topic. That's intentional - the *topic* is supplied later, through the `Task` descriptions and the `report_inputs` dictionary, which keeps these agent definitions reusable for any topic.

### Agents 1-6: the section writers

The first six agents each own one section of the final report:

1. **Research & Evidence Analyst** - gathers facts, statistics, recent examples, and case studies that the other writers can lean on.
2. **Introduction Writer** - frames the topic and hooks the reader.
3. **Core Argument Analyst** - states the report's central thesis and the reasoning behind it.
4. **Multi-Dimensional Analyst** - explores how that thesis plays out across different dimensions of the issue.
5. **Counter-Argument & Critique Specialist** - raises the strongest objections and responds to them.
6. **Synthesis & Conclusion Writer** - ties everything together and proposes recommendations.

In [7]:
fact_finder_agent = Agent(
    role="Research & Evidence Analyst",
    goal=(
        "Compile accurate, concrete facts, statistics, recent real-world examples, and "
        "case studies on the report's topic, clearly noting approximate dates, locations, "
        "and sources so that the other writers can rely on them."
    ),
    backstory=(
        "You have spent years gathering data on environmental and disaster-related issues "
        "for policy reports. You favour credible sources such as government and UN agency "
        "reports, peer-reviewed studies, and reputable news coverage, and you always note "
        "roughly when and where something happened so claims can be checked later."
    ),
    llm=llm,
    verbose=True,
)

intro_writer_agent = Agent(
    role="Introduction Writer",
    goal=(
        "Write an engaging introduction that frames the report's topic, explains why it "
        "matters right now, and previews what the rest of the report will cover."
    ),
    backstory=(
        "You open long-form reports for a living. Readers decide in the first paragraph "
        "whether to keep reading, so you hook them with a concrete detail before zooming "
        "out to the bigger picture."
    ),
    llm=llm,
    verbose=True,
)

core_argument_agent = Agent(
    role="Core Argument Analyst",
    goal=(
        "State the report's central thesis clearly and lay out the main line of reasoning "
        "and evidence that supports it."
    ),
    backstory=(
        "You think in terms of claim, reasons, and evidence. You are good at distilling a "
        "complex issue down to one clear, defensible thesis and explaining why it holds up."
    ),
    llm=llm,
    verbose=True,
)

multi_dimension_agent = Agent(
    role="Multi-Dimensional Analyst",
    goal=(
        "Show how the report's core argument plays out across several different "
        "dimensions - for example environmental, economic, social or public-health, and "
        "political or governance dimensions - so readers see the full picture."
    ),
    backstory=(
        "You specialise in systems thinking: you map how one issue ripples outward into "
        "other areas of life, and you organise that complexity into clearly labelled "
        "sections so it stays easy to follow."
    ),
    llm=llm,
    verbose=True,
)

counter_argument_agent = Agent(
    role="Counter-Argument & Critique Specialist",
    goal=(
        "Present the strongest good-faith objections or alternative viewpoints to the "
        "report's core argument, and respond to each of them fairly."
    ),
    backstory=(
        "You play devil's advocate professionally. You state the opposing view as "
        "persuasively as its proponents would, and then give a measured, evidence-based "
        "response rather than a dismissive one."
    ),
    llm=llm,
    verbose=True,
)

synthesis_conclusion_agent = Agent(
    role="Synthesis & Conclusion Writer",
    goal=(
        "Pull together the introduction, core argument, multi-dimensional analysis, "
        "evidence, and counter-arguments into one coherent synthesis, restate the overall "
        "takeaway, and give a short list of concrete recommendations."
    ),
    backstory=(
        "You write the closing section of reports. You are good at noticing the thread "
        "that runs through everything that came before and turning it into a memorable, "
        "actionable ending."
    ),
    llm=llm,
    verbose=True,
)

### Agent 7: the Fact-Checker & Humanizing Editor

This agent has a dual job, both described in `skills/humanizer/SKILL.md`: it cross-checks the claims in each section against the evidence the research agent gathered, and it rewrites the section so it doesn't read like typical AI output.

Instead of repeating those rules in every task description, we embed the entire contents of `humanizer_instructions` directly into this agent's `backstory` using an ordinary Python f-string. Because this happens once, when the agent object is created, the same instructions are automatically available no matter which section this agent is asked to review later. We will reuse this single agent for six separate review tasks - one per section.

In [8]:
verifier_humanizer_agent = Agent(
    role="Fact-Checker & Humanizing Editor",
    goal=(
        "Independently check each section of the report for factual accuracy against the "
        "evidence gathered, then rewrite it so it reads naturally and avoids robotic, "
        "repetitive AI-style phrasing."
    ),
    backstory=(
        "You are a meticulous copy-editor and fact-checker. You always cross-check claims "
        "against the evidence you are given before changing anything else, and you follow "
        "the house style guide below very closely whenever you rewrite a section.\n\n"
        f"{humanizer_instructions}"
    ),
    llm=llm,
    verbose=True,
)

### Agent 8: the Managing Editor (combiner)

Once every section has been individually verified and humanized, something still needs to stitch them into one document with a title, an executive summary, and consistent terminology from start to finish. That's this agent's job.

In [9]:
combiner_agent = Agent(
    role="Managing Editor",
    goal=(
        "Combine the verified, humanized sections into one cohesive, well-structured "
        "Markdown report with smooth transitions, consistent terminology, and a single "
        "unified voice."
    ),
    backstory=(
        "You assemble individually-written sections into a finished report. You smooth "
        "over abrupt transitions, remove duplicated points, make sure headings and "
        "terminology are consistent throughout, and add a short title and a one-paragraph "
        "executive summary at the very top."
    ),
    llm=llm,
    verbose=True,
)

### Agent 9: the Publishing Specialist (Word document writer)

The final agent has one job: hand the finished report to the `save_report_as_word_document` tool we defined in Step 7, exactly as written, and report back the confirmation message. We attach the tool to the agent via the `tools=[...]` argument, which makes it available for the agent to call while completing its task.

In [10]:
doc_writer_agent = Agent(
    role="Publishing Specialist",
    goal=(
        "Take the final, combined report and use the available tool to save it as a "
        "polished Word document, then return the tool's confirmation message."
    ),
    backstory=(
        "You are the final step before a report goes out the door. You do not rewrite "
        "content - your job is to faithfully hand the finished Markdown report to the "
        "document-generation tool and confirm that the file was saved."
    ),
    tools=[save_report_as_word_document],
    llm=llm,
    verbose=True,
)

## 9. Defining the tasks

A `Task` describes one unit of work: what to do (`description`), what a good result looks like (`expected_output`), which `agent` should do it, and which earlier tasks' outputs it needs as `context`. When a task lists other tasks in `context`, CrewAI automatically passes those tasks' outputs to the agent as additional information, which is how work flows from one agent to the next.

Notice that the `description` and `expected_output` strings below contain `{topic}`, `{audience}`, and `{tone}` placeholders. CrewAI fills these in from the `report_inputs` dictionary we defined in Step 5 when we call `crew.kickoff(inputs=report_inputs)` - this is the templating mechanism mentioned earlier.

We'll define the tasks in the order the crew will run them. First, the research task and the five section-writing tasks:

- The **research task** runs first, with no dependencies, and produces the evidence dossier.
- Each **section task** lists `research_task` in its `context`, so every section writer has access to that evidence.
- The **multi-dimensional** and **counter-argument** tasks also depend on `core_argument_task`, since they build on the thesis it states.
- The **synthesis task** depends on *all* of the previous section tasks, since its job is to tie them together.

In [11]:
research_task = Task(
    description=(
        "Research the topic: '{topic}'. Compile a structured evidence dossier containing:\n"
        "1. Key facts and statistics from credible reports (for example IPCC, government "
        "ministries, UN agencies, or peer-reviewed studies).\n"
        "2. At least three recent real-world examples or disasters that illustrate the "
        "issue, each with an approximate date and location.\n"
        "3. One or two short case studies describing impacts and/or responses in more "
        "depth.\n"
        "Write with {audience} in mind, and for each item briefly note what it illustrates."
    ),
    expected_output=(
        "A Markdown document with three sub-sections titled 'Key Facts & Statistics', "
        "'Recent Examples', and 'Case Studies', each containing several concrete, "
        "specific bullet points."
    ),
    agent=fact_finder_agent,
)

intro_task = Task(
    description=(
        "Using the evidence dossier as background, write the Introduction section of a "
        "report on '{topic}'. Explain what the issue is, why it matters right now, and "
        "briefly preview what the rest of the report will cover. Write in a {tone} tone "
        "for {audience}."
    ),
    expected_output=(
        "An Introduction section in Markdown (roughly 3-5 paragraphs), starting with the "
        "heading '## Introduction'."
    ),
    agent=intro_writer_agent,
    context=[research_task],
)

core_argument_task = Task(
    description=(
        "Drawing on the evidence dossier, write the Core Argument section of the report "
        "on '{topic}'. State the central thesis in one or two clear sentences near the "
        "start of the section, then explain the main reasoning and evidence that "
        "supports it. Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "A Core Argument section in Markdown starting with the heading '## Core "
        "Argument', containing a clearly stated thesis and supporting reasoning."
    ),
    agent=core_argument_agent,
    context=[research_task],
)

multi_dimension_task = Task(
    description=(
        "Building on the core argument and the evidence dossier, write a "
        "Multi-Dimensional Discussion section for the report on '{topic}'. Discuss how "
        "the core argument plays out across several relevant dimensions - for example "
        "environmental, economic, social or public-health, and political or governance "
        "dimensions - using a clearly labelled sub-heading for each dimension. Write in "
        "a {tone} tone for {audience}."
    ),
    expected_output=(
        "A 'Multi-Dimensional Discussion' section in Markdown starting with '## "
        "Multi-Dimensional Discussion', containing at least three labelled sub-sections "
        "(use '### ' headings)."
    ),
    agent=multi_dimension_agent,
    context=[research_task, core_argument_task],
)

counter_argument_task = Task(
    description=(
        "Considering the core argument and the evidence dossier, write a "
        "Counter-Arguments section for the report on '{topic}'. Present at least two of "
        "the strongest good-faith objections or alternative viewpoints, and follow each "
        "one with a brief, fair, evidence-based response. Write in a {tone} tone for "
        "{audience}."
    ),
    expected_output=(
        "A 'Counter-Arguments' section in Markdown starting with '## Counter-Arguments', "
        "containing at least two objection/response pairs."
    ),
    agent=counter_argument_agent,
    context=[research_task, core_argument_task],
)

synthesis_task = Task(
    description=(
        "Read the introduction, core argument, multi-dimensional discussion, evidence "
        "dossier, and counter-arguments for the report on '{topic}'. Write a Synthesis & "
        "Conclusion section that ties these threads together, restates the overall "
        "takeaway in your own words, and ends with a short bulleted list of two to four "
        "concrete recommendations. Write in a {tone} tone for {audience}."
    ),
    expected_output=(
        "A 'Synthesis & Conclusion' section in Markdown starting with '## Synthesis & "
        "Conclusion', ending with a bulleted list of recommendations."
    ),
    agent=synthesis_conclusion_agent,
    context=[research_task, intro_task, core_argument_task, multi_dimension_task, counter_argument_task],
)

## 10. Defining the verification tasks

Next we define six review tasks - one for each section, including the evidence dossier itself. Every one of these tasks is assigned to the *same* `verifier_humanizer_agent`, but each task gives it a different pair of inputs in its `context`: the original evidence dossier (so it has something to fact-check against) and the specific section it should review and rewrite.

Because the fact-checking checklist and the humanizing style rules already live in this agent's `backstory` (Step 8), the task descriptions below can stay short - they just need to tell the agent *which* section to work on and what heading to preserve.

In [12]:
verify_evidence_task = Task(
    description=(
        "Review the evidence dossier you were given for the report on '{topic}'. Check "
        "that the figures, dates, and named events are plausible and internally "
        "consistent. Then rewrite it following the fact-checking and humanizing rules in "
        "your backstory. Keep the same three sub-headings ('Key Facts & Statistics', "
        "'Recent Examples', 'Case Studies')."
    ),
    expected_output=(
        "A revised version of the evidence dossier in Markdown, with the same three "
        "sub-headings, written in natural prose and bullet points and free of robotic "
        "phrasing."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task],
)

verify_intro_task = Task(
    description=(
        "Review the Introduction section you were given for the report on '{topic}', "
        "cross-checking any factual claims against the evidence dossier. Then rewrite it "
        "following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output="A revised Introduction section in Markdown, starting with '## Introduction'.",
    agent=verifier_humanizer_agent,
    context=[research_task, intro_task],
)

verify_core_argument_task = Task(
    description=(
        "Review the Core Argument section you were given for the report on '{topic}', "
        "cross-checking any factual claims against the evidence dossier. Then rewrite it "
        "following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output="A revised Core Argument section in Markdown, starting with '## Core Argument'.",
    agent=verifier_humanizer_agent,
    context=[research_task, core_argument_task],
)

verify_multi_dimension_task = Task(
    description=(
        "Review the Multi-Dimensional Discussion section you were given for the report "
        "on '{topic}', cross-checking any factual claims against the evidence dossier. "
        "Then rewrite it following the fact-checking and humanizing rules in your "
        "backstory. Keep its sub-headings."
    ),
    expected_output=(
        "A revised Multi-Dimensional Discussion section in Markdown, starting with '## "
        "Multi-Dimensional Discussion' and keeping its '### ' sub-headings."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, multi_dimension_task],
)

verify_counter_argument_task = Task(
    description=(
        "Review the Counter-Arguments section you were given for the report on "
        "'{topic}', cross-checking any factual claims against the evidence dossier. Then "
        "rewrite it following the fact-checking and humanizing rules in your backstory."
    ),
    expected_output="A revised Counter-Arguments section in Markdown, starting with '## Counter-Arguments'.",
    agent=verifier_humanizer_agent,
    context=[research_task, counter_argument_task],
)

verify_synthesis_task = Task(
    description=(
        "Review the Synthesis & Conclusion section you were given for the report on "
        "'{topic}', cross-checking any factual claims against the evidence dossier. Then "
        "rewrite it following the fact-checking and humanizing rules in your backstory. "
        "Keep its bulleted list of recommendations."
    ),
    expected_output=(
        "A revised Synthesis & Conclusion section in Markdown, starting with '## "
        "Synthesis & Conclusion' and ending with a bulleted list of recommendations."
    ),
    agent=verifier_humanizer_agent,
    context=[research_task, synthesis_task],
)

## 11. Defining the combine and publish tasks

The last two tasks finish the pipeline:

- The **combine task** gives the Managing Editor agent the six verified, humanized sections as `context` and asks it to produce one final Markdown report with a title and executive summary.
- The **document task** gives the Publishing Specialist agent that final report as `context` and asks it to call the `save_report_as_word_document` tool to write it to disk.

In [13]:
combine_task = Task(
    description=(
        "Combine the verified and humanized sections for the report on '{topic}' - "
        "evidence dossier (for your own reference only), introduction, core argument, "
        "multi-dimensional discussion, counter-arguments, and synthesis & conclusion - "
        "into one cohesive Markdown report. Add a single '# ' title at the very top that "
        "reflects the topic, followed by a short executive-summary paragraph, and then "
        "the sections in this order: Introduction, Core Argument, Multi-Dimensional "
        "Discussion, Evidence & Case Studies, Counter-Arguments, Synthesis & Conclusion. "
        "Smooth over any abrupt transitions and make sure terminology is consistent "
        "throughout."
    ),
    expected_output=(
        "A single, complete Markdown report beginning with a '# ' title and an executive "
        "summary, followed by the six sections in the specified order, with consistent "
        "headings and terminology."
    ),
    agent=combiner_agent,
    context=[
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
    ],
)

doc_task = Task(
    description=(
        "Take the final combined report exactly as written and use the "
        "save_report_as_word_document tool to save it as a Word document named "
        "'environmental_security_india_report.docx'. Pass the full report text to the "
        "tool without summarising, shortening, or rewriting it, and return the tool's "
        "confirmation message as your final answer."
    ),
    expected_output=(
        "The confirmation message returned by the save_report_as_word_document tool, "
        "including the absolute path of the saved .docx file."
    ),
    agent=doc_writer_agent,
    context=[combine_task],
)

## 12. Assembling the crew

A `Crew` brings together the agents and tasks we've defined and decides how to run them. We use `Process.sequential`, which runs the tasks one after another in the order given, passing each task's `context` outputs along as it goes. This mirrors how a real editorial team might work: research first, then drafting, then review, then final assembly and publication.

Setting `verbose=True` makes the crew print a detailed log of which agent is working on which task, and what it produces, as it runs.

In [14]:
crew = Crew(
    agents=[
        fact_finder_agent,
        intro_writer_agent,
        core_argument_agent,
        multi_dimension_agent,
        counter_argument_agent,
        synthesis_conclusion_agent,
        verifier_humanizer_agent,
        combiner_agent,
        doc_writer_agent,
    ],
    tasks=[
        research_task,
        intro_task,
        core_argument_task,
        multi_dimension_task,
        counter_argument_task,
        synthesis_task,
        verify_evidence_task,
        verify_intro_task,
        verify_core_argument_task,
        verify_multi_dimension_task,
        verify_counter_argument_task,
        verify_synthesis_task,
        combine_task,
        doc_task,
    ],
    process=Process.sequential,
    verbose=True,
)

## 13. Running the crew

`crew.kickoff(inputs=report_inputs)` starts the pipeline. CrewAI substitutes `{topic}`, `{audience}`, and `{tone}` in every task description and expected output using the dictionary we defined in Step 5, then runs the fourteen tasks in order.

This involves at least fourteen separate calls to the language model (one per task, plus extra calls whenever the Publishing Specialist needs to call the Word-document tool), so expect this cell to take a few minutes to finish. Because `verbose=True`, you'll see a running log of each agent's thinking and output as it goes - this is a good way to understand what each agent actually contributes.

In [16]:
result = await crew.kickoff_async(inputs=report_inputs)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8713e5fb-a51f-4e17-b91a-d4fe0a464be8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Research the topic: 'Environmental Security in India'. Compile a structured evidence dossier             │
│  containing:                                                                                                    │
│  1. Key facts and statistics from credible reports (for example IPCC, government ministries, UN agencies, or    │
│  peer-reviewed studies).                                                                                        │
│  2. At least three recent real-world examples or disasters that illustrate the issue, each with an approximate  │
│  date and location.                                                                                             │
│  3. One or two short case studies describing impacts and/or responses in more depth.                            │
│  Write with policy analysts, journalists, and informed general readers in mind, and for each item briefly note  │
│  what it illustrates.                                                                                           │
│  ID: ef314313-5eee-4a25-ad75-bfec8e5737f6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research & Evidence Analyst                                                                             │
│                                                                                                                 │
│  Task: Research the topic: 'Environmental Security in India'. Compile a structured evidence dossier             │
│  containing:                                                                                                    │
│  1. Key facts and statistics from credible reports (for example IPCC, government ministries, UN agencies, or    │
│  peer-reviewed studies).                                                                                        │
│  2. At least three recent real-world examples or disasters that illustrate the issue, each with an approximate  │
│  date and location.                                                                                             │
│  3. One or two short case studies describing impacts and/or responses in more depth.                            │
│  Write with policy analysts, journalists, and informed general readers in mind, and for each item briefly note  │
│  what it illustrates.                                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Research & Evidence Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Environmental Security in India: An Evidence Dossier                                                         │
│                                                                                                                 │
│  **Prepared by:** Research & Evidence Analyst                                                                   │
│  **Target Audience:** Policy Analysts, Journalists, and Informed General Readers                                │
│                                                                                                                 │
│  Environmental security in India is a critical dimension of national security, directly impacting human         │
│  survival, economic stability, and social cohesion. This dossier compiles verified data, recent real-world      │
│  events, and detailed case studies to outline the systemic environmental threats facing the country.            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Key Facts & Statistics                                                                                     │
│                                                                                                                 │
│  *   **Severe Groundwater Depletion and Water Stress**                                                          │
│      *   **The Data:** According to the *Composite Water Management Index (CWMI)* report published by NITI      │
│  Aayog (Government of India, 2018/2019), approximately 600 million Indians face high-to-extreme water stress.   │
│  The report projected that 21 major cities—including New Delhi, Bengaluru, Chennai, and Hyderabad—would         │
│  severely deplete their groundwater tables, threatening the water security of 100 million people.               │
│      *   **What it illustrates:** The acute threat to human security, urban viability, and agricultural         │
│  sustainability posed by unregulated groundwater extraction and inadequate water conservation infrastructure.   │
│                                                                                                                 │
│  *   **High Vulnerability to Sea-Level Rise and Coastal Flooding**                                              │
│      *   **The Data:** The *Intergovernmental Panel on Climate Change (IPCC) Sixth Assessment Report (Working   │
│  Group II, 2022)* identified India as one of the countries most vulnerable to climate-induced sea-level rise.   │
│  The report notes that by mid-century, over 35 million people in India's coastal areas could face annual        │
│  coastal flooding, with major economic hubs like Mumbai, Chennai, and Kolkata at extreme risk.                  │
│      *   **What it illustrates:** The systemic risk to critical infrastructure, coastal populations, and        │
│  metropolitan economies from rising global temperatures.                                                        │
│                                                                                                                 │
│  *   **Widespread Land Degradation and Desertification*

...

╭───────────────────────────────────────────────── ❌ LLM Error ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  LLM Call Failed                                                                                                │
│  Error: Google Gemini API error: 429 - You exceeded your current quota, please check your plan and billing      │
│  details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To    │
│  monitor your current usage, head to: https://ai.dev/rate-limit.                                                │
│  * Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit:     │
│  20, model: gemini-3.5-flash                                                                                    │
│  Please retry in 32.726340652s.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Failure ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Failed                                                                                                    │
│  Name: Review the Introduction section you were given for the report on 'Environmental Security in India',      │
│  cross-checking any factual claims against the evidence dossier. Then rewrite it following the fact-checking    │
│  and humanizing rules in your backstory.                                                                        │
│  Agent: Fact-Checker & Humanizing Editor                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── Crew Failure ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Failed                                                                                          │
│  Name: crew                                                                                                     │
│  ID: 8713e5fb-a51f-4e17-b91a-d4fe0a464be8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 32.726340652s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3.5-flash', 'location': 'global'},

## 14. Viewing the results

`crew.kickoff(...)` returns a `CrewOutput` object. Its `.raw` attribute holds the text produced by the *last* task in the pipeline - in our case, that's the confirmation message from the Publishing Specialist's tool call, telling us where the `.docx` file was saved.

The full combined report (before it was handed to the document-writing tool) is the output of `combine_task`, which we can reach via `combine_task.output.raw`. We display both below: the tool's confirmation message, and the final report rendered as Markdown so you can read it directly in the notebook.

In [ ]:
print("Publishing Specialist's confirmation:")
print(result.raw)

print("\n--- Final combined report ---\n")
final_report_markdown = combine_task.output.raw
display(Markdown(final_report_markdown))

## 15. Ideas for extending this notebook

- **Change the topic**: edit `report_inputs` in Step 5 and re-run the notebook from that cell onwards (no agent or task definitions need to change).
- **Use a different model or provider**: edit the single `llm = LLM(...)` cell in Step 4 - for example, point it at `gpt-4o-mini` (OpenAI) or `groq/llama-3.3-70b-versatile` (Groq) using the keys already loaded from `.env`.
- **Give the research agent real-time web search**: install `crewai-tools` and add a search tool (such as `SerperDevTool`, which needs its own API key) to `fact_finder_agent`'s `tools=[...]` list, so its facts and examples are grounded in live search results rather than the model's training data.
- **Tighten or loosen the humanizer rules**: edit `skills/humanizer/SKILL.md` - because that file is read fresh each time you run Step 6, any changes you make there will flow straight into the Fact-Checker & Humanizing Editor's `backstory` the next time you run the notebook.
- **Tune how much the crew "thinks"**: adjust `temperature` in Step 4 (lower for more consistent, conservative writing; higher for more varied phrasing), or change `Process.sequential` to `Process.hierarchical` in Step 12 to let a manager agent decide the running order itself.